# RAG evaluation

This notebook displays the 25-question benchmark for the five PDFs in `data/`. The scripts use the parser and prompt from `eda_workbook.ipynb` plus `ingest.py` and `retrieval.py`. Labels, metric definitions, and limitations are in [evaluation/README.md](../evaluation/README.md).

In [ ]:
from pathlib import Path
import json, subprocess, sys
import pandas as pd
from IPython.display import display, Markdown

root = Path.cwd() if (Path.cwd() / "evaluation").is_dir() else Path.cwd().parent
assert (root / "evaluation/benchmark.json").exists(), "Run from the repo root or notebooks/"
print(root)

## Retrieval

Run this cell after `python -m pip install -r evaluation/requirements.txt`. The first run downloads MPNet and builds the Chroma index; later runs reuse the cached index.

In [ ]:
subprocess.run([sys.executable, str(root / "evaluation/run_retrieval.py")], cwd=root, check=True)

In [ ]:
retrieval = json.loads((root / "evaluation/results/retrieval.json").read_text())
display(pd.DataFrame([retrieval["summary"]]).T.rename(columns={0: "value"}))
rows = retrieval["queries"]
display(pd.DataFrame([{"id": q["id"], "query": q["query"], "gold": ", ".join(q["relevant_papers"]), "retrieved": ", ".join(q["retrieved_papers"]), "evidence": f"{sum(q['evidence_hits'])}/{len(q['evidence_hits'])}"} for q in rows]))

## Fresh answers

Start Ollama and pull `llama3.2` in a terminal (`ollama serve`, then `ollama pull llama3.2`). The next cell calls the notebook prompt with the retrieved passages. It resumes completed answers if rerun. The saved notebook outputs in `eda_workbook.ipynb` came from another corpus and are not scored here.

In [ ]:
subprocess.run([sys.executable, str(root / "evaluation/run_answers.py")], cwd=root, check=True)

In [ ]:
subprocess.run([sys.executable, str(root / "evaluation/check_answers.py")], cwd=root, check=True)
checks = json.loads((root / "evaluation/results/answer_checks.json").read_text())
display(pd.DataFrame([checks["summary"]]).T.rename(columns={0: "value"}))

In [ ]:
answers = json.loads((root / "evaluation/results/answers.json").read_text())["answers"]
selected_id = "q01"  # change to any ID from q01 to q25
row = next(a for a in answers if a["id"] == selected_id)
display(Markdown(f"### {selected_id}: {row['query']}\n\n{row['answer']}\n\n**Sources**\n\n{row['source_list']}"))
for passage in row["passages"]:
    display(Markdown(f"**{passage['paper_id']} · {passage['section']}**\n\n> {passage['text'][:1000]}..."))

## Review rubric

For each answer, check that its citations point to supporting passages, all material claims are correct, each requested fact is covered or explicitly missing, and the answer contains a direct quote when required by issue #5. For q24–q25, check that the model declines to invent KPMG or market data. Record the manual judgments separately from retrieval metrics; the presence of a citation does not establish factual accuracy.

The completed per-query review is in [evaluation/results/answer_review.md](../evaluation/results/answer_review.md), and improvement opportunities are in [evaluation/results/findings.md](../evaluation/results/findings.md).